# Runbook 5.3.a: Encryption vs. hashing

**Skill:** Mitigate Risk · **Maps to:** 5.3.A · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** In secure systems, credentials like passwords should never be stored in their original plaintext form, nor should they be encrypted (which implies they could be decrypted if the key is compromised). Instead, they should be hashed using a one-way cryptographic function combined with a unique, random value called a **salt**. The salt ensures that identical passwords result in different digests, defeating precomputed lookup attacks (like rainbow tables).

Your task is to implement a password hashing and verification module.

You must implement the function `solve(action: str, password: str, param: str)` to handle two actions:
1. **Hash a password:** Given action `"hash"`, a plaintext password, and a salt, compute the SHA-256 hash of the concatenated string `salt + password`. Return a combined string of the format `salt$digest` (where `digest` is the hex-encoded hash).
2. **Verify a password:** Given action `"verify"`, a plaintext password, and a stored hash string in the format `salt$digest`, extract the salt and digest, compute the salted hash of the password, and verify if it matches the expected digest. Use a constant-time comparison helper like `hmac.compare_digest` to prevent timing side-channel attacks.

**Lesson objective.** Distinguish encryption from hashing, explain the one-way property that makes them suited for different jobs, and choose the correct control when a scenario requires either confidentiality or integrity verification.

### What comes in
- `action`: Either `"hash"` or `"verify"`
- `password`: The plaintext password
- `param`: Either the `salt` (if hashing) or the `stored_hash` of format `salt$digest` (if verifying)

### What must come out
- If hashing: A string of format `salt$digest`
- If verifying: A boolean indicating whether the password matches the digest

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```python
solve("hash", "myPassword", "somesalt")
# -> "somesalt$4f24ef3001859bf9016e7a750ef90333d833890ea70c99a6cfbf6e568ea46a9a"

solve("verify", "myPassword", "somesalt$4f24ef3001859bf9016e7a750ef90333d833890ea70c99a6cfbf6e568ea46a9a")
# -> True

solve("verify", "wrongPassword", "somesalt$4f24ef3001859bf9016e7a750ef90333d833890ea70c99a6cfbf6e568ea46a9a")
# -> False
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(action: str, password: str, param: str) -> Union[str, bool]
```

#### Input
- `action`: A string, either `"hash"` or `"verify"`.
- `password`: A string, the plaintext password to hash or verify.
- `param`: 
  - If `action` is `"hash"`: this is the salt (string) to be prepended to the password.
  - If `action` is `"verify"`: this is the stored hash (string) of format `salt$digest` to verify against.

#### Output
- If `action` is `"hash"`: returns `salt$digest` where `digest` is the hex-encoded SHA-256 hash of `salt + password`.
- If `action` is `"verify"`: returns a boolean (`True` if matches, `False` otherwise).

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network.
- Standard library only.
- Safe verification should use a constant-time comparison helper like `hmac.compare_digest`.

</details>

## 3. Build it

Fill in `solve` below. Run the cell (Shift+Enter) after each change.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
"""Starter stub for 5.3.a."""
from typing import Union


def solve(action: str, password: str, param: str) -> Union[str, bool]:
    """Perform salted hashing or verification of passwords."""
    raise NotImplementedError("replace with the working reference solution")


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

import hashlib


def test_hash_simple():
    salt = "random_salt"
    pwd = "password123"
    expected_digest = hashlib.sha256((salt + pwd).encode("utf-8")).hexdigest()
    expected_hash = f"{salt}${expected_digest}"
    assert impl.solve("hash", pwd, salt) == expected_hash


def test_verify_correct():
    salt = "s3cr3t_s4lt"
    pwd = "mySuperSecretPassword!"
    expected_digest = hashlib.sha256((salt + pwd).encode("utf-8")).hexdigest()
    stored_hash = f"{salt}${expected_digest}"
    assert impl.solve("verify", pwd, stored_hash) is True


def test_verify_incorrect():
    salt = "s3cr3t_s4lt"
    pwd = "mySuperSecretPassword!"
    expected_digest = hashlib.sha256((salt + pwd).encode("utf-8")).hexdigest()
    stored_hash = f"{salt}${expected_digest}"
    assert impl.solve("verify", "wrongPassword!", stored_hash) is False


def test_verify_invalid_format():
    assert impl.solve("verify", "password", "no_dollar_sign_here") is False


def test_empty_password_and_salt():
    salt = ""
    pwd = ""
    expected_digest = hashlib.sha256((salt + pwd).encode("utf-8")).hexdigest()
    expected_hash = f"{salt}${expected_digest}"
    assert impl.solve("hash", pwd, salt) == expected_hash
    assert impl.solve("verify", pwd, expected_hash) is True


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Call `solve` on an input you make up, and predict the answer first.

- [ ] Your prediction matched

## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it